<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit3/session-13-secure-mcp-server/notebook.ipynb)


# Session 13 — Build and secure an MCP server

**Wed 30 Sep 2026 · 2h**

You build the part of an MCP server that says no. One read-only tool, one
narrow argument, and a guard it consults **before** it fetches anything. Then
you point the same reading at a real comprehended on-chain surface, from dated
recordings, and place every activity in the lowest lane that can do it.

## Three lanes, and you only need the first

| Lane | What you need | What it does |
|---|---|---|
| **Recorded** | nothing | Real responses, captured and dated. Everything below runs on these. |
| **Public read-only** | a network connection | The live surface, no key, no account. Browse, comprehend, derive. Stops before any signature. |
| **Fork rehearsal** | the instructor | The signing loop, on a disposable fork. Optional, and nobody needs it to finish this session. |

Nothing in this notebook reaches the network, in any lane, unless you opt in at
section 8. Nothing in it can spend money, in any lane. The one tool that signs
does it with a throwaway key on a local fork, and it cannot reach mainnet at all.


In [20]:
print("fixture" in globals(), "check" in globals())

False True


In [21]:
from bootcamp_agent.checks import check, review


## 1. The smallest honest server

A server is a contract with an outside world that does not trust you and that
you do not trust either. The smallest honest one declares three things:

- **what it exposes** — one tool, `read_page`, with one argument;
- **what that tool may touch** — an allowlist of hosts, written down;
- **what it refuses** — and a refusal that names the rule that fired.

Excess scope is the failure this shape prevents. A `fetch` tool that takes any
URL is not one capability, it is every capability the network offers, handed to
whatever text arrives in the model's context. Narrow the argument, declare the
hosts, and the tool can only do the job it was added for.


In [22]:
# The surface this server declares. One tool, one argument, read-only.
TOOL = {
    "name": "read_page",
    "description": "Fetch one page from an allowed host and return its text.",
    "input_schema": {
        "type": "object",
        "properties": {"url": {"type": "string"}},
        "required": ["url"],
    },
    "read_only": True,
}

# The allowlist is configuration, not an argument. A caller cannot widen it.
ALLOWED_DOMAINS = ("example.com", "docs.python.org")

print(TOOL["name"], "takes", list(TOOL["input_schema"]["properties"]))
print("read-only:", TOOL["read_only"])
print("allowed hosts:", ", ".join(ALLOWED_DOMAINS), "(and their subdomains)")


read_page takes ['url']
read-only: True
allowed hosts: example.com, docs.python.org (and their subdomains)


## 2. Exercise: the fetch guard

**Context.** `read_page` cannot fetch a URL and then decide whether it should
have. By then the request has left, the internal service has answered, and the
answer is in the model's context. The decision has to happen on the string,
before the socket opens.

Write `fetch_guard(url) -> {"allowed": bool, "reason": str}`. Three rules, in
this order, and the first one that fires decides:

| # | Rule | Refuses |
|---|---|---|
| 1 | **scheme** | anything that is not `http` or `https` — `file://`, `gopher://`, `ftp://` |
| 2 | **address** | a host that IS an address, unless the address is public: `10.x`, `192.168.x`, `172.16–172.31.x`, `127.x`, `localhost`, `::1`, `169.254.x` including `169.254.169.254`, `0.0.0.0` |
| 3 | **allowlist** | a host that is not `ALLOWED_DOMAINS` or a subdomain of one |

**Instructions.**

1. `reason` starts with one of three codes — `scheme`, `not-public`,
   `not-allowlisted` — and anything after a colon is your own detail.
2. Every string gets a verdict. A guard that raises on a bad URL hands the
   decision back to a caller with no rules to decide it with.
3. Two things the check will try, because they are how real guards fail:
   `http://2130706433/` and `http://0x7f000001/` are `127.0.0.1` written as one
   number, and `https://evil-example.com` merely ends with an allowed domain.
4. Read the host with a URL parser, never off the string. In
   `https://example.com@evil-example.net/`, the host is `evil-example.net`.

`ipaddress` and `urllib.parse` are in the standard library and do most of this
for you. `ip_address(host).is_private` knows where `172.16.0.0/12` stops; a
prefix test does not.


In [23]:
import re
import socket
from ipaddress import ip_address
from urllib.parse import urlsplit

ALLOWED_SCHEMES = ("http", "https")
ALLOWED_DOMAINS = ("example.com", "docs.python.org")
LOCAL_NAMES = {"localhost", "ip6-localhost", "ip6-loopback"}
LOCAL_SUFFIXES = (".localhost", ".local", ".localdomain", ".internal")


def _as_address(host: str):
    """Return an ip_address if the host is one in any spelling, else None."""
    try:
        return ip_address(host)
    except ValueError:
        pass
    if re.fullmatch(r"[0-9a-fA-Fx.]+", host):
        try:  # decimal, hex, octal and shorthand IPv4: 2130706433, 0x7f.1, 0177.0.0.1, 127.1
            return ip_address(socket.inet_ntoa(socket.inet_aton(host)))
        except (OSError, ValueError):
            pass
    return None


def fetch_guard(url: str) -> dict:
    """Decide, before any request is made, whether this URL may be fetched."""
    # rule 1 — the scheme
    try:
        parts = urlsplit(url)
        scheme = parts.scheme.lower()
        hostname = parts.hostname
    except ValueError:
        return {"allowed": False, "reason": "not-allowlisted: URL could not be parsed"}

    if scheme not in ALLOWED_SCHEMES:
        return {"allowed": False, "reason": f"scheme: {scheme or '(none)'} is not http or https"}

    host = (hostname or "").rstrip(".").lower()
    if not host:
        return {"allowed": False, "reason": "not-allowlisted: URL has no hostname"}

    # rule 2 — the address, if the host is one, or a name for this machine
    if host in LOCAL_NAMES or host.endswith(LOCAL_SUFFIXES):
        return {"allowed": False, "reason": f"not-public: {host} is a name for this machine or a private network"}
    addr = _as_address(host)
    if addr is not None:
        if getattr(addr, "ipv4_mapped", None):
            addr = addr.ipv4_mapped
        if not addr.is_global:
            return {"allowed": False, "reason": f"not-public: {host} is {addr}, a non-public address"}

    # rule 3 — the allowlist: the domain itself or a subdomain of it
    for domain in ALLOWED_DOMAINS:
        if host == domain or host.endswith("." + domain):
            return {"allowed": True, "reason": f"{host} is on the allowlist"}

    return {"allowed": False, "reason": f"not-allowlisted: {host} is not on the allowlist"}

**Expected output** (your wording after each colon will differ)

```
ALLOW  https://example.com/pricing                  example.com is on the allowlist
ALLOW  https://api.example.com/v1/status            api.example.com is on the allowlist
REFUSE http://169.254.169.254/latest/meta-data/     not-public: 169.254.169.254 is ...
REFUSE http://2130706433/                           not-public: 2130706433 is 127.0.0.1 ...
REFUSE https://evil-example.com/spec.json           not-allowlisted: evil-example.com ...
REFUSE file:///etc/passwd                           scheme: file is not http or https
✅ ch13-e4 passed
```

The check drives your guard with thirty-one URLs of its own, and it reads the
reason as well as the verdict. Refused for the wrong reason is a failure here:
the reason is what the next person acts on.


In [24]:
check("ch13-e4", fetch_guard)


✅ ch13-e4 passed


True

## 3. The redirect, and why one check is not a guard

A guard that runs once runs on the URL you were given, not on the URL you end
up fetching. A public host answers `302` and names the next hop, and the next
hop is the metadata service. Nothing about the first URL was wrong.

So the guard runs on **every hop**, and the number of hops is capped. The cell
below walks a recorded chain. Nothing is fetched: the hops are a list, and the
guard is a function of a string.


In [25]:
# A redirect chain, met the way a fetcher meets one: one hop at a time.
RECORDED_REDIRECTS = [
    "https://docs.example.com/spec",              # 302 ->
    "https://example.com/v2/spec",                # 302 ->
    "http://169.254.169.254/latest/meta-data/",   # the hop that was the point
]

for number, hop in enumerate(RECORDED_REDIRECTS, start=1):
    verdict = fetch_guard(hop)
    print(f"hop {number}: {'ALLOW ' if verdict['allowed'] else 'REFUSE'}  {hop}")
    if not verdict["allowed"]:
        print(f"         stopped: {verdict['reason']}")
        break


hop 1: ALLOW   https://docs.example.com/spec
hop 2: ALLOW   https://example.com/v2/spec
hop 3: REFUSE  http://169.254.169.254/latest/meta-data/
         stopped: not-public: 169.254.169.254 is 169.254.169.254, a non-public address


## 4. The applied case: a real surface, and what a tool list proves

The rest of this session reads a surface somebody else built and comprehended:
Orquestra's catalogue of Solana programs, through Gecko. Every response below
is a **recording**, stamped with the day it was captured.

A surface tells you which tools it has. That is all a list is: sixteen names.
It says nothing about what any of them does when you call it, and nothing about
whether anybody has ever called it successfully.


In [38]:
import json
import pathlib
import os

def fixture(name: str) -> dict:
    return json.loads(
        (pathlib.Path("fixtures") / f"{name}.json").read_text(encoding="utf-8")
    )


tools = fixture("orquestra-tools")
print(f"\n{tools['_provenance']['count']} tools, protocol {tools['_provenance']['protocol']}\n")
for name in tools["tools"]:
    print(" ", name)
print("\nis evidence of    :", tools["_provenance"]["is_evidence_of"])
print("is NOT evidence of:", tools["_provenance"]["is_not_evidence_of"])



16 tools, protocol 2025-11-25

  start
  find_start
  list_programs
  comprehend_program
  prepare_purchase
  try_purchase
  list_stores
  plan_payment
  plan_swap
  verify_signed_transaction
  submit_transaction
  read_accounts
  prepare_instruction
  derive_ata
  derive_pda
  program_lifecycle

is evidence of    : which tools the surface exposes, and nothing about what they do when called
is NOT evidence of: that every listed tool was called or tested. Only read-only calls were.


## 5. Browse: what the surface says about a store

`list_stores` reads storefronts from one node's view of the chain, on the day
it was recorded. Read the `note` the surface itself returns. It is unusually
careful, and the care is the lesson: a menu is not an authorization.


In [39]:
stores = fixture("list-stores-recorded")
print("\nnote:", stores["note"][:180], "...\n")
for store in stores["stores"]:
    print(store["address"][:12], "...")
    for product in store["products"]:
        print(f"    {product['name'][:28]:30} {product['price_ui']:>8} {product['mint_note']}")



note: a MENU read from one node's view of the chain, not an authorization: the authorities listed here are what the store accounts SAY, and a purchase still verifies everything before an ...

gi7tTEQVZjhg ...
    Jägermeister                        0.5 USDC
23hqxfyG5uDG ...
    Shot                                0.5 USDC


## 6. Comprehend: the accounts an instruction needs, and where each came from

`find_start` returns a derive plan: the accounts a call needs, in derivation
order, each tagged with **where the value came from**. `extracted` means the
surface read it from the program. A `FLAGGED` gap means it could not, and it
says so instead of guessing.

That tag is the same idea as the reason on your refusal. An account list
without provenance is a guess with good formatting; a refusal without a reason
is a shrug with good formatting.


In [40]:
start = fixture("find-start-recorded")
plan = start["starts"][0]["derive_plan"]
print(f"\n{len(plan)} accounts, in derivation order:\n")
for step in plan:
    print(f"  {step['account']:28} {step['provenance']}")

tiers = sorted({step["provenance"] for step in plan})
print("\nprovenance tiers present:", tiers)



3 accounts, in derivation order:

  receipts                     extracted
  sender_token_account         extracted
  recipient_token_account      extracted

provenance tiers present: ['extracted']


## 7. Exercise: which lane can do what

**Context.** Four lanes, and one of them is `never`. Being able to place an
activity in the lowest lane that can do it is what stops a demo from quietly
becoming a live spend.

**Instructions.**

1. For each activity, name the **lowest** lane that can do it.
2. Choose from `recorded`, `public-read-only`, `fork`, `never`.
3. One of these is `never`, and it is not a trick.


In [41]:
lanes = {
    "read the 16 tool names": "recorded",   # a fixture already holds this
    "derive the accounts one instruction needs": "recorded",
    "see the price a store charges today": "public-read-only",
    "sign anything": "fork",
    "spend real money on mainnet": "never",
}
for activity, lane in lanes.items():
    print(f"{lane or '(unset)':18} {activity}")

recorded           read the 16 tool names
recorded           derive the accounts one instruction needs
public-read-only   see the price a store charges today
fork               sign anything
never              spend real money on mainnet


**Expected output**

```
recorded           read the 16 tool names
recorded           derive the accounts one instruction needs
public-read-only   see the price a store charges today
fork               sign anything
never              spend real money on mainnet
✅ ch13-e2 passed
```


In [42]:
check("ch13-e2", lanes)


✅ ch13-e2 passed


True

## 8. Optional: the live surface, read-only

The hosted surface was open to a read-only probe on 3 September 2026: no
credential, protocol `2025-11-25`, sixteen tools. This cell is **opt-in**
because it reaches the network, and a notebook that silently makes network
calls is a notebook you cannot trust offline.

```bash
GECKO_LIVE=1 uv run jupyter lab
```

Connect a client with:

```bash
claude mcp add --transport http gecko https://mcp.geckovision.tech/orquestra/mcp
```

Then ask it to list the stores and compare today's prices against the recorded
fixture above. **Stop there.** Nothing below browsing is part of this lane.


In [43]:
if os.environ.get("GECKO_LIVE") != "1":
    print("skipped: the live lane is opt-in. Set GECKO_LIVE=1 to enable it.")
    print("Everything this session teaches is already done, on the recorded lane.")
else:
    print("Live lane enabled. Use your MCP client, not this notebook:")
    print("  claude mcp add --transport http gecko https://mcp.geckovision.tech/orquestra/mcp")
    print("Then: list the stores, and compare a price against the fixture above.")
    print("Read-only only. Do not call prepare_purchase, try_purchase or submit_transaction.")


skipped: the live lane is opt-in. Set GECKO_LIVE=1 to enable it.
Everything this session teaches is already done, on the recorded lane.


## 9. Optional: the fork rehearsal, with the instructor

If the instructor has provisioned a disposable fork, this is where the signing
loop runs: `prepare_purchase`, then `try_purchase`, then read the receipt and
judge it by **what moved**, not by whether the call returned.

The key that signs there is created only after the endpoint proves it is a
fork, it is funded by a cheatcode, and it cannot reach mainnet. If you are
working through this alone, skip it. Nothing later depends on it.


## 10. Exercise: the safety checklist

**Context.** Self-attested, and the last box is the one that matters. If you
stayed in the recorded and read-only lanes, the honest answer to
`ran_the_fork_lane` is `False`, and saying so is the point.

**Instructions.**

1. Tick each box only if it is true for what you actually did.
2. `ran_the_fork_lane` is `True` or `False`. Most people answer `False`.
3. Write why nothing you ran could have spent real money, in your own words.


In [44]:
checklist = {
    "no_credentials_entered": True,
    "url_not_committed": True,
    "provenance_tier_named": True,
    "mode_for_an_unknown_api": True,
    "ran_the_fork_lane": False,
    "why_nothing_could_spend": "Everything ran on recorded fixtures, the live lane was skipped, and nothing here holds a key or a signer, so no transaction could be signed or sent.",
}
for box, state in checklist.items():
    mark = "[x]" if state is True else "[ ]" if state is False else "..."
    print(f"{mark} {box}")

[x] no_credentials_entered
[x] url_not_committed
[x] provenance_tier_named
[x] mode_for_an_unknown_api
[ ] ran_the_fork_lane
... why_nothing_could_spend


**Expected output** (yours may differ in wording, not in shape):

```
[x] no_credentials_entered
[x] url_not_committed
[x] provenance_tier_named
[x] mode_for_an_unknown_api
[ ] ran_the_fork_lane
... why_nothing_could_spend
✅ ch13-e1 passed
```


In [45]:
check("ch13-e1", checklist)


✅ ch13-e1 passed


True

## Exit ticket

Homework: add an **external tools** section to your project instructions —
allowed hosts, data handling, who approves what. Then write the paragraph you
would send a colleague who asks "can I let an agent fetch any URL for me?" Name
the rule that fires for `http://169.254.169.254/`, and say what your guard does
on a redirect.


## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.


In [46]:
review("ch13")


ch13: 3/3 passed  ·  300/300 marks


True

In [47]:
import bootcamp_agent, bootcamp_agent.checks as c
print([n for n in dir(c) if not n.startswith("_")])
print([n for n in dir(bootcamp_agent) if not n.startswith("_")])

['AnswerParseError', 'Any', 'CH06_QUERIES', 'CHECKS', 'CORPUS_DIR', 'Callable', 'Checker', 'ExerciseCheckFailed', 'FakeLLM', 'Path', 'REPO_ROOT', 'RESULTS', 'ResearchAnswer', 'STRICT_ENV', 'ToolError', 'UnknownCheck', 'annotations', 'check', 'json', 'load_cases', 'load_corpus', 'os', 'parse_research_answer', 'register', 'retrieve', 'review', 'run_evals']
['agent', 'checks', 'config', 'curriculum', 'depth_checks', 'documents', 'evals', 'hints', 'llm', 'retrieval', 'schema', 'session_checks', 'tools', 'week0_checks']


In [48]:
import pathlib
from bootcamp_agent import checks as c

root = pathlib.Path(c.REPO_ROOT)
for p in root.rglob("*"):
    if ".git" in p.parts or ".venv" in p.parts or "node_modules" in p.parts:
        continue
    if "fixture" in p.name.lower() or "orquestra" in p.name.lower() or "recorded" in p.name.lower():
        print(p.relative_to(root))
    elif p.suffix in {".py", ".ipynb"}:
        try:
            if "fixture" in p.read_text(encoding="utf-8", errors="ignore"):
                print("mentions fixture:", p.relative_to(root))
        except OSError:
            pass

cookbook\fixtures
mentions fixture: cookbook\advanced\08_anti_poisoning.ipynb
mentions fixture: cookbook\advanced\09_program_graph_find_start.ipynb
mentions fixture: cookbook\integrations\06_capstone_meets_gecko.ipynb
mentions fixture: cookbook\integrations\07_two_apis_one_assistant.ipynb
mentions fixture: cookbook\integrations\11_autonomous_petshop.py
mentions fixture: cookbook\introduction\01_welcome_to_gecko.ipynb
mentions fixture: cookbook\introduction\02_comprehend_an_openapi.ipynb
cookbook\introduction\03_recorded_mode_calls.ipynb
mentions fixture: cookbook\introduction\05_read_the_report.ipynb
mentions fixture: demos\07_the_coach_in_a_chat.ipynb
mentions fixture: demos\09_rag_on_your_laptop.ipynb
demos\fixtures
demos\fixtures\rag-recorded.json
mentions fixture: depth\04-secure-and-reliable\solutions\notebook.ipynb
depth\06-graph-rag\fixtures
mentions fixture: depth\06-graph-rag\notebook.ipynb
mentions fixture: depth\06-graph-rag\solutions\notebook.ipynb
final_assignment\fixtures

In [49]:
import pathlib
from bootcamp_agent import checks as c

root = pathlib.Path(c.REPO_ROOT)
for folder in ["src", "notebooks", "data", "fixtures", "recorded"]:
    d = root / folder
    if not d.exists():
        continue
    for p in d.rglob("*"):
        n = p.name.lower()
        if "fixture" in n or "orquestra" in n or "recorded" in n:
            print(p.relative_to(root))
        elif p.suffix in {".py", ".ipynb"}:
            try:
                if "def fixture" in p.read_text(encoding="utf-8", errors="ignore"):
                    print("defines fixture:", p.relative_to(root))
            except OSError:
                pass
print("done")

defines fixture: src\bootcamp_agent\week0_checks\dsa.py
done
